In [1]:
# ============================================================================
# Drought characterization — Rio Parana basin
# Method: Run Theory (Yevjevich, 1967)
# Index: SPEI-6
# Metrics: Frequency, Duration (median), Severity (median)
# Definition: event starts when SPEI < -1 for >= 2 consecutive months
#             event ends when SPEI >= 0
# Reference: Spinoni et al. (2014), Tabari et al. (2021)
# Scenarios: Historical (1985-2014), SSP126 (2041-2070), SSP585 (2041-2070)
# Models: MPI-ESM1-2-HR, UKESM1-0-LL, GFDL-ESM4, IPSL-CM6A-LR, MRI-ESM2-0
# ============================================================================

import os
import numpy as np
import xarray as xr
import netCDF4 as nc
from pathlib import Path

# --- CONFIGURATION ---
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "rio_parana"

path_in  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/3.outputs_SPEI_R/5.baseline_106_6ac_OF/Rio_Parana/"
path_out = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/"


# ============================================================================
# FUNCTION: Calculate drought metrics for a single time series
# ============================================================================
def calc_drought_metrics(spei_ts):
    n           = len(spei_ts)
    in_event    = False
    consec_neg  = 0
    event_start = None
    durations   = []
    severities  = []

    for t in range(n):
        val = spei_ts[t]

        if np.isnan(val):
            in_event   = False
            consec_neg = 0
            continue

        if not in_event:
            if val < -1:
                consec_neg += 1
                if consec_neg >= 2:
                    in_event    = True
                    event_start = t - 1
            else:
                consec_neg = 0
        else:
            if val >= 0:
                duration = (t - 1) - event_start + 1
                severity = np.sum(np.abs(spei_ts[event_start:t]))
                durations.append(duration)
                severities.append(severity)
                in_event   = False
                consec_neg = 0

    if in_event:
        duration = (n - 1) - event_start + 1
        severity = np.sum(np.abs(spei_ts[event_start:n]))
        durations.append(duration)
        severities.append(severity)

    frequency = len(durations)
    if frequency == 0:
        return frequency, np.nan, np.nan

    return frequency, np.median(durations), np.median(severities)


# ============================================================================
# FUNCTION: Save drought metrics as NetCDF
# ============================================================================
def save_netcdf(freq_grid, dur_grid, sev_grid, lon, lat,
                basin, scenario, model, output_dir):
    output_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
    ds_out      = nc.Dataset(output_file, "w", format="NETCDF4")

    ds_out.createDimension("lat", len(lat))
    ds_out.createDimension("lon", len(lon))

    lat_var       = ds_out.createVariable("lat", "f4", ("lat",))
    lat_var.units = "degrees_north"
    lat_var[:]    = lat

    lon_var       = ds_out.createVariable("lon", "f4", ("lon",))
    lon_var.units = "degrees_east"
    lon_var[:]    = lon

    freq_save = np.where(np.isnan(freq_grid), -9999, freq_grid)
    dur_save  = np.where(np.isnan(dur_grid),  -9999, dur_grid)
    sev_save  = np.where(np.isnan(sev_grid),  -9999, sev_grid)

    freq_var           = ds_out.createVariable("frequency", "f4", ("lat", "lon"),
                                                fill_value=-9999)
    freq_var.units     = "count"
    freq_var.long_name = "Number of drought events"
    freq_var[:]        = freq_save

    dur_var           = ds_out.createVariable("duration", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    dur_var.units     = "months"
    dur_var.long_name = "Median drought duration"
    dur_var[:]        = dur_save

    sev_var           = ds_out.createVariable("severity", "f4", ("lat", "lon"),
                                               fill_value=-9999)
    sev_var.units     = "-"
    sev_var.long_name = "Median drought severity (sum of |SPEI| per event)"
    sev_var[:]        = sev_save

    ds_out.description = (f"Drought characteristics — {basin} basin, "
                          f"{scenario}, {model}. "
                          f"Run theory (Yevjevich 1967): "
                          f"start SPEI < -1 for >= 2 months, end SPEI >= 0.")
    ds_out.index       = "SPEI-6"
    ds_out.basin       = basin
    ds_out.scenario    = scenario
    ds_out.model       = model

    ds_out.close()
    print(f"    Saved: {output_file}")


# ============================================================================
# FUNCTION: Calculate ensemble median across 5 models
# ============================================================================
def calc_ensemble(basin, scenario, models, output_dir):
    freq_stack = []
    dur_stack  = []
    sev_stack  = []

    for model in models:
        nc_file = f"{output_dir}drought_metrics_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)

        freq = np.where(ds["frequency"].values == -9999, np.nan, ds["frequency"].values.astype(float))
        dur  = np.where(ds["duration"].values  == -9999, np.nan, ds["duration"].values.astype(float))
        sev  = np.where(ds["severity"].values  == -9999, np.nan, ds["severity"].values.astype(float))

        freq_stack.append(freq)
        dur_stack.append(dur)
        sev_stack.append(sev)

        lon = ds["lon"].values
        lat = ds["lat"].values
        ds.close()

    freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
    dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
    sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)

    save_netcdf(freq_ensemble, dur_ensemble, sev_ensemble,
                lon, lat, basin, scenario, "ensemble", output_dir)

    print(f"\n  Ensemble summary — {scenario}:")
    for name, grid in [("FREQUENCY", freq_ensemble),
                       ("DURATION",  dur_ensemble),
                       ("SEVERITY",  sev_ensemble)]:
        valid = grid[~np.isnan(grid)]
        print(f"    {name}: min={valid.min():.2f}, "
              f"max={valid.max():.2f}, "
              f"median={np.median(valid):.2f}")


# ============================================================================
# MAIN LOOP — SCENARIOS x MODELS
# ============================================================================
for scenario in scenarios:

    out_dir = f"{path_out}{scenario}/"
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario.upper()} | BASIN: {basin.upper()}")
    print(f"{'='*60}")

    for model in models:

        print(f"\n  Processing: {model}")

        nc_file = f"{path_in}{scenario}/spei6_{basin}_{scenario}_{model}.nc"
        ds      = xr.open_dataset(nc_file)
        spei    = ds["SPEI_6"].values
        lon     = ds["lon"].values
        lat     = ds["lat"].values
        dims    = ds["SPEI_6"].dims
        ds.close()

        spei = np.where(spei == -9999, np.nan, spei)

        if dims[0] == "time":
            spei = np.transpose(spei, (1, 2, 0))

        n_lon, n_lat, _ = spei.shape

        freq_grid = np.full((n_lon, n_lat), np.nan)
        dur_grid  = np.full((n_lon, n_lat), np.nan)
        sev_grid  = np.full((n_lon, n_lat), np.nan)

        for i in range(n_lon):
            for j in range(n_lat):
                ts = spei[i, j, :]
                if np.all(np.isnan(ts)):
                    continue
                (freq_grid[i, j],
                 dur_grid[i, j],
                 sev_grid[i, j]) = calc_drought_metrics(ts)

        for name, grid in [("FREQUENCY", freq_grid),
                           ("DURATION",  dur_grid),
                           ("SEVERITY",  sev_grid)]:
            valid = grid[~np.isnan(grid)]
            print(f"    {name}: min={valid.min():.2f}, "
                  f"max={valid.max():.2f}, "
                  f"median={np.median(valid):.2f}")

        save_netcdf(freq_grid, dur_grid, sev_grid,
                    lon, lat, basin, scenario, model, out_dir)

    print(f"\n  Computing ensemble median — {scenario}")
    calc_ensemble(basin, scenario, models, out_dir)


print(f"\n{'='*60}")
print(f"COMPLETED — Rio Parana basin, 5 models, 3 scenarios")
print(f"Output: {path_out}")
print(f"{'='*60}")


SCENARIO: HISTORICAL | BASIN: RIO_PARANA

  Processing: MPI-ESM1-2-HR
    FREQUENCY: min=4.00, max=17.00, median=9.00
    DURATION: min=4.00, max=47.00, median=8.00
    SEVERITY: min=3.65, max=46.88, median=9.36
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/historical/drought_metrics_rio_parana_historical_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=3.00, max=16.00, median=9.00
    DURATION: min=4.00, max=34.00, median=8.50
    SEVERITY: min=3.84, max=44.75, median=9.59
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/historical/drought_metrics_rio_parana_historical_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=3.00, max=16.00, median=9.00
    DURATION: min=4.00, max=31.00, median=9.00
    SEVERITY: min=4.48, max=38.44, median=10.77
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/R

/tmp/ipykernel_1991325/2139998709.py:155: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:156: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:157: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    FREQUENCY: min=3.00, max=17.00, median=11.00
    DURATION: min=4.00, max=85.00, median=9.00
    SEVERITY: min=4.18, max=108.94, median=10.01
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp126/drought_metrics_rio_parana_ssp126_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=1.00, max=18.00, median=10.00
    DURATION: min=4.00, max=352.00, median=10.00
    SEVERITY: min=3.86, max=418.36, median=11.70
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp126/drought_metrics_rio_parana_ssp126_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=5.00, max=19.00, median=12.00
    DURATION: min=5.00, max=56.00, median=10.00
    SEVERITY: min=5.40, max=59.43, median=11.59
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp126/drought_metrics_rio_parana_ssp126_GFDL-ESM4.nc

  Processing

/tmp/ipykernel_1991325/2139998709.py:155: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:156: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:157: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


    FREQUENCY: min=1.00, max=18.00, median=10.00
    DURATION: min=4.00, max=337.00, median=12.00
    SEVERITY: min=4.64, max=423.56, median=13.37
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp585/drought_metrics_rio_parana_ssp585_MPI-ESM1-2-HR.nc

  Processing: UKESM1-0-LL
    FREQUENCY: min=1.00, max=20.00, median=10.00
    DURATION: min=4.50, max=360.00, median=14.50
    SEVERITY: min=4.33, max=459.09, median=16.61
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp585/drought_metrics_rio_parana_ssp585_UKESM1-0-LL.nc

  Processing: GFDL-ESM4
    FREQUENCY: min=1.00, max=17.00, median=11.00
    DURATION: min=5.50, max=356.00, median=13.00
    SEVERITY: min=5.95, max=419.21, median=15.49
    Saved: /data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/ssp585/drought_metrics_rio_parana_ssp585_GFDL-ESM4.nc

  Proces

/tmp/ipykernel_1991325/2139998709.py:155: RuntimeWarning: All-NaN slice encountered
  freq_ensemble = np.nanmedian(np.stack(freq_stack, axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:156: RuntimeWarning: All-NaN slice encountered
  dur_ensemble  = np.nanmedian(np.stack(dur_stack,  axis=0), axis=0)
/tmp/ipykernel_1991325/2139998709.py:157: RuntimeWarning: All-NaN slice encountered
  sev_ensemble  = np.nanmedian(np.stack(sev_stack,  axis=0), axis=0)


In [2]:
# ============================================================================
# Verification — Rio Parana basin — SPEI-6
# ============================================================================

import numpy as np
import xarray as xr
from pathlib import Path

path_out  = "/data/brussel/vo/000/bvo00033/vsc11346/SPEI_R/7.Caracterizacion_sequia/SPEI-6/2.Outputs/Rio_Parana/"
models    = ["MPI-ESM1-2-HR", "UKESM1-0-LL", "GFDL-ESM4", "IPSL-CM6A-LR", "MRI-ESM2-0", "ensemble"]
scenarios = ["historical", "ssp126", "ssp585"]
basin     = "rio_parana"

print("=" * 60)
print("NETCDF VERIFICATION — Rio Parana basin — SPEI-6")
print("=" * 60)

all_ok = True

for scenario in scenarios:
    print(f"\n--- {scenario.upper()} ---")
    for model in models:
        nc_file = Path(f"{path_out}{scenario}/drought_metrics_{basin}_{scenario}_{model}.nc")

        if not nc_file.exists():
            print(f"  ❌ MISSING: {nc_file.name}")
            all_ok = False
            continue

        ds   = xr.open_dataset(nc_file)
        freq = np.where(ds["frequency"].values == -9999, np.nan, ds["frequency"].values.astype(float))
        dur  = np.where(ds["duration"].values  == -9999, np.nan, ds["duration"].values.astype(float))
        sev  = np.where(ds["severity"].values  == -9999, np.nan, ds["severity"].values.astype(float))
        ds.close()

        n_valid   = np.sum(~np.isnan(freq))
        shape_ok  = freq.ndim == 2
        pixels_ok = (np.sum(~np.isnan(freq)) == np.sum(~np.isnan(dur)) == np.sum(~np.isnan(sev)))
        values_ok = (np.all(freq[~np.isnan(freq)] >= 0) and
                     np.all(dur[~np.isnan(dur)]   >= 0) and
                     np.all(sev[~np.isnan(sev)]   >= 0))
        no_inf    = not (np.any(np.isinf(freq)) or np.any(np.isinf(dur)) or np.any(np.isinf(sev)))

        status = "✅" if (shape_ok and pixels_ok and values_ok and no_inf) else "❌"
        if status == "❌":
            all_ok = False

        print(f"  {status} {model}")
        print(f"       Shape: {freq.shape} | Valid pixels: {n_valid} | "
              f"Freq=[{freq[~np.isnan(freq)].min():.1f},{freq[~np.isnan(freq)].max():.1f}] | "
              f"Dur=[{dur[~np.isnan(dur)].min():.1f},{dur[~np.isnan(dur)].max():.1f}] | "
              f"Sev=[{sev[~np.isnan(sev)].min():.2f},{sev[~np.isnan(sev)].max():.2f}]")

print(f"\n{'='*60}")
print("✅ ALL FILES VERIFIED — SPEI-6" if all_ok else "❌ SOME FILES HAVE ISSUES")
print(f"{'='*60}")

NETCDF VERIFICATION — Rio Parana basin — SPEI-6

--- HISTORICAL ---
  ✅ MPI-ESM1-2-HR
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[4.0,17.0] | Dur=[4.0,47.0] | Sev=[3.65,46.88]
  ✅ UKESM1-0-LL
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[3.0,16.0] | Dur=[4.0,34.0] | Sev=[3.84,44.75]
  ✅ GFDL-ESM4
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[3.0,16.0] | Dur=[4.0,31.0] | Sev=[4.48,38.44]
  ✅ IPSL-CM6A-LR
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[4.0,16.0] | Dur=[3.0,45.5] | Sev=[3.86,51.80]
  ✅ MRI-ESM2-0
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[5.0,16.0] | Dur=[4.0,18.5] | Sev=[4.55,22.10]
  ✅ ensemble
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[4.0,14.0] | Dur=[5.0,34.0] | Sev=[5.53,34.03]

--- SSP126 ---
  ✅ MPI-ESM1-2-HR
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[3.0,17.0] | Dur=[4.0,85.0] | Sev=[4.18,108.94]
  ✅ UKESM1-0-LL
       Shape: (41, 58) | Valid pixels: 1652 | Freq=[1.0,18.0] | Dur=[4.0,352.0] | Sev=[3.86,418.36]
  